# Week 5 – Builder (Cemre Gönenç): bigger OCR test set + hard plates

**Goal:** OCR **≥ 80%**, and a fixed **"hard plates" set of 30** to measure progress every week.

1. Download a Turkish plate dataset where every **character** is labelled (Roboflow `metehan-yasar/plaka-cript`, CC BY 4.0).
   Reading the character boxes left→right gives the true plate text → hundreds of labelled plates with no typing.
2. Measure our OCR on this bigger set (not just our 50 plates).
3. Pick the 30 hardest plates → `ANPR/hard_plates/`.
4. Try sharpening and thresholding on them, keep what helps.

Needs a free **Roboflow API key** (roboflow.com → Settings → API Keys). Runtime → **T4 GPU**. Run cells one by one.

In [ ]:
!pip install -q paddlepaddle paddleocr roboflow

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
else:
    !cd anpr-project-en && git pull -q
os.chdir('/content/anpr-project-en')
print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ANPR'
RESULTS = f'{D}/results'; os.makedirs(RESULTS, exist_ok=True)
import sys, pandas as pd
sys.path.insert(0, 'ocr')
from plate_reader import PlateReader, ENHANCE_MODES
!python ocr/test_plate_text.py | tail -1
!python ocr/test_week5.py | tail -1

## 1) Download character-labelled Turkish plates

In [ ]:
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('Roboflow API key: '))
CHAR_SETS = [('metehan-yasar', 'plaka-cript'), ('metehan-yasar', 'plaka-tanima-all')]
char_yamls = {}
for ws, proj in CHAR_SETS:
    try:
        p = rf.workspace(ws).project(proj)
        v = max(p.versions(), key=lambda v: int(v.version))
        ds = v.download('yolov8', location=f'/content/datasets/{proj}')
        char_yamls[proj] = f'{ds.location}/data.yaml'
        print('OK', proj, 'version', v.version)
    except Exception as e:
        print('FAILED', proj, '->', e)

## 2) Build the text test set (true text from the character boxes)

In [ ]:
V2 = f'{D}/plates_v2'
frames = []
for proj, y in char_yamls.items():
    out = f'{V2}/{proj}'
    !python ocr/build_text_testset.py --data "$y" --out "$out"
    df = pd.read_csv(f'{out}/labels.csv'); df['path'] = out + '/' + df['filename']; df['source'] = proj
    frames.append(df)
big = pd.concat(frames, ignore_index=True).drop_duplicates('plate_text')
print(len(big), 'unique plates')
big.sample(min(10, len(big)), random_state=0)[['filename', 'plate_text', 'source']]

In [ ]:
import cv2, matplotlib.pyplot as plt
s = big.sample(min(12, len(big)), random_state=1)
plt.figure(figsize=(15, 5))
for i, (_, r) in enumerate(s.iterrows()):
    plt.subplot(3, 4, i + 1); plt.imshow(cv2.cvtColor(cv2.imread(r['path']), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title(r['plate_text'])
plt.tight_layout(); plt.show()
print('Check a few: if the titles do not match the plates, the character order is wrong – tell Claude.')

## 3) OCR accuracy on the bigger set (Week 3 setting: gray + 2x + fixes)
Uses up to 300 plates to keep it under ~5 minutes.

In [ ]:
import time
reader = PlateReader(gray=True, scale=2.0, fix=True)
test = big.sample(min(300, len(big)), random_state=0).reset_index(drop=True)
t0 = time.time()
out = [reader.read(p) for p in test['path']]
test['read'] = [o['text'] for o in out]; test['score'] = [o['score'] for o in out]; test['valid'] = [o['valid'] for o in out]
test['ok'] = test['read'] == test['plate_text']
print(f"Bigger set: {test['ok'].sum()}/{len(test)} = {100 * test['ok'].mean():.1f}%  ({time.time() - t0:.0f} s)")
print('Target 80%:', 'PASSED' if test['ok'].mean() >= 0.8 else 'NOT YET')

## 4) Hard plates set (30): wrong readings first, then the lowest-confidence ones

In [ ]:
import shutil
hard = pd.concat([test[~test['ok']].sort_values('score'), test[test['ok']].sort_values('score')]).head(30)
HARD = f'{D}/hard_plates'; os.makedirs(HARD, exist_ok=True)
for i, (_, r) in enumerate(hard.iterrows(), 1):
    shutil.copy(r['path'], f'{HARD}/hard_{i:02d}.jpg')
hard_labels = pd.DataFrame({'filename': [f'hard_{i:02d}.jpg' for i in range(1, len(hard) + 1)],
                            'plate_text': hard['plate_text'].values, 'source': hard['source'].values})
hard_labels.to_csv(f'{HARD}/labels.csv', index=False)
print(len(hard_labels), 'hard plates saved to', HARD, '| wrong among them:', int((~hard['ok']).sum()))
print('This set stays FIXED – rerun section 5 every week to measure progress.')

## 5) Sharpening / thresholding on the hard set and on the whole set

In [ ]:
hl = pd.read_csv(f'{HARD}/labels.csv')
rows = []
configs = [('Week 3 (no extra)', None, ()), ('sharpen', 'sharpen', ()), ('clahe', 'clahe', ()),
           ('otsu threshold', 'otsu', ()), ('adaptive threshold', 'adaptive', ()),
           ('fallback: sharpen→clahe→adaptive', None, ('sharpen', 'clahe', 'adaptive'))]
for name, enh, fb in configs:
    reader.enhance, reader.fallback = enh, fb
    h_ok = sum(reader.read(f'{HARD}/{f}')['text'] == t for f, t in zip(hl['filename'], hl['plate_text']))
    a_ok = sum(reader.read(p)['text'] == t for p, t in zip(test['path'], test['plate_text']))
    rows.append({'Setting': name, 'Hard 30 correct': h_ok, 'Hard %': round(100 * h_ok / len(hl), 1),
                 f'All {len(test)} correct': a_ok, 'All %': round(100 * a_ok / len(test), 1)})
res = pd.DataFrame(rows); res

**How to choose:** pick the setting with the best **All %** (it must not hurt normal plates).
The fallback setting only tries the extra cleanups when the first reading does not fit the plate pattern, so it usually helps hard plates without hurting easy ones.
Put the winner in `app/pipeline.py` → `PlateReader(gray=True, scale=2.0, fix=True, enhance=..., fallback=...)`.

In [ ]:
from datetime import date
best = res.sort_values(['All %', 'Hard %'], ascending=False).iloc[0]
log = f'{RESULTS}/hard_plates_progress.csv'
row = pd.DataFrame([{'date': date.today(), 'setting': best['Setting'], 'hard_%': best['Hard %'], 'all_%': best['All %'], 'n_all': len(test)}])
row.to_csv(log, mode='a', header=not os.path.exists(log), index=False)
test.to_csv(f'{RESULTS}/week5_bigger_set.csv', index=False); res.to_csv(f'{RESULTS}/week5_enhance.csv', index=False)
print(pd.read_csv(log))
print(f"\nRow for docs/experiments.md (OCR table):")
print(f"| {date.today()} | B | PaddleOCR + {best['Setting']} | {len(test)} plates (plaka-cript) | – | {best['All %']}% | hard-30: {best['Hard %']}% |")

## 6) Conclusion (fill in)
- Accuracy on the bigger set: … %  (target 80%)
- Best extra cleanup: …
- Hard-30 this week: … %  (keep the log in `ANPR/results/hard_plates_progress.csv`)
- Still failing: …